# 3- Score Model Outputs

This notebook demonstrates the scoring module. The scoring rubric is implemented in `src/scoring.py`; this notebook applies it to `raw_outputs.csv` and saves `scored_outputs.csv`.

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
import pandas as pd

from src.config import RAW_OUTPUTS_FILENAME, SCORED_OUTPUTS_FILENAME, get_data_dir
from src.scoring import score_outputs

DATA_DIR = get_data_dir(PROJECT_DIR)
RAW_OUTPUTS_PATH = DATA_DIR / RAW_OUTPUTS_FILENAME
SCORED_OUTPUTS_PATH = DATA_DIR / SCORED_OUTPUTS_FILENAME

pd.set_option("display.max_colwidth", 160)

print("Raw outputs path:", RAW_OUTPUTS_PATH)
print("Scored outputs path:", SCORED_OUTPUTS_PATH)

## Load generated responses

In [ ]:
raw_df = pd.read_csv(RAW_OUTPUTS_PATH)

print(f"Loaded {len(raw_df):,} generated responses")
display(raw_df.head())

## Check response balance

In [ ]:
display(raw_df["model_type"].value_counts().rename("count").to_frame())
display(raw_df.groupby(["domain", "model_type"]).size().unstack(fill_value=0))
display(raw_df.groupby(["prompt_type", "model_type"]).size().unstack(fill_value=0))

responses_per_prompt = raw_df.groupby("prompt_id")["model_type"].nunique()
print(f"Unique prompts: {raw_df['prompt_id'].nunique():,}")
print(f"Prompts with both model types: {(responses_per_prompt == 2).sum():,}")

## Apply rule-based scoring

The scorer assigns leaning, neutrality, hedging, refusal, and response-length outcomes.

In [ ]:
scored_df = score_outputs(raw_df, output_path=SCORED_OUTPUTS_PATH)

print(f"Scored responses: {len(scored_df):,}")
display(scored_df[[
    "prompt_id", "model_type", "leaning_score", "neutrality_score",
    "hedging_score", "refusal_score", "response_length"
]].head(10))

## Inspect score distributions

In [ ]:
score_columns = [
    "leaning_score",
    "neutrality_score",
    "hedging_score",
    "refusal_score",
    "response_length",
]

display(scored_df.groupby("model_type")[score_columns].describe().round(2))

for col in ["leaning_score", "neutrality_score", "hedging_score", "refusal_score"]:
    print(f"Distribution for {col}")
    display(pd.crosstab(scored_df[col], scored_df["model_type"], margins=True))

## Save check

In [ ]:
saved = pd.read_csv(SCORED_OUTPUTS_PATH)
print(f"Saved scored outputs to: {SCORED_OUTPUTS_PATH}")
print("Shape:", saved.shape)
display(saved.head())